# schema-tuned-sql — Remote NVIDIA Tesla T4 Runner

Executes execution-verified SFT (LoRA 1.5B FP16 and QLoRA 3B 4-bit NF4), dev/test evaluation with 2,000-resample bootstrap CIs and paired McNemar tests, general-SQL catastrophic forgetting checks, and T4 serving benchmarks on an NVIDIA Tesla T4 (sm_75, 16 GB VRAM).

In [ ]:
import os
import subprocess

import torch

REPO_URL = "https://github.com/adityavsb/schema-tuned-sql.git"
REPO_DIR = "schema-tuned-sql"
OUT_DIR = "t4_results"
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

assert torch.cuda.is_available(), "CUDA device not detected."
cc_major, cc_minor = torch.cuda.get_device_capability(0)
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
print(
    f"GPU: {torch.cuda.get_device_name(0)} (sm_{cc_major}{cc_minor}, {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB VRAM)"
)
!nvidia-smi

In [ ]:
# 1. Build DuckDB warehouse and run execution-verified distillation (2,000 pairs, 8 Qs/call)
!PYTHONPATH={REPO_DIR} python {REPO_DIR}/warehouse/generate_data.py --seed 42 --db-path {REPO_DIR}/warehouse/analytics.duckdb
!PYTHONPATH={REPO_DIR} python {REPO_DIR}/src/datagen/generator.py --config {REPO_DIR}/configs/data_gen.yaml --mode live

In [ ]:
# 2. Fine-tune S1 (Qwen2.5-Coder-1.5B FP16 LoRA) and S2 (Qwen2.5-Coder-3B 4-bit QLoRA) on T4
!PYTHONPATH={REPO_DIR} python {REPO_DIR}/src/train/sft.py --config {REPO_DIR}/configs/sft_qwen15b_lora.yaml
!PYTHONPATH={REPO_DIR} python {REPO_DIR}/src/train/sft.py --config {REPO_DIR}/configs/sft_qwen3b_qlora.yaml

In [ ]:
# 3. Scaling curve (250 / 500 / 1k / 2k) and LoRA rank (8 / 16 / 64) ablations
for sz in [250, 500, 1000, 2000]:
    subprocess.run(
        [
            "python",
            f"{REPO_DIR}/src/train/sft.py",
            "--config",
            f"{REPO_DIR}/configs/sft_qwen3b_qlora.yaml",
            "--subset-size",
            str(sz),
            "--seed",
            "42",
        ],
        check=True,
    )

for r in [8, 16, 64]:
    subprocess.run(
        [
            "python",
            f"{REPO_DIR}/src/train/sft.py",
            "--config",
            f"{REPO_DIR}/configs/sft_qwen3b_qlora.yaml",
            "--lora-r",
            str(r),
            "--seed",
            "42",
        ],
        check=True,
    )

In [ ]:
# 4. Run T4 Serving Benchmark (Batch 1, 8, 32) and Cost Break-Even Model
!PYTHONPATH={REPO_DIR} python {REPO_DIR}/src/serve/bench.py --config {REPO_DIR}/configs/serve.yaml --mode cuda_live --out {OUT_DIR}/serving_cost_analysis_t4.json